In [9]:
import geopandas as gpd
from shapely.geometry import Point
import pandas as pd
from alive_progress import alive_bar
from collections import Counter

In [10]:
neighborhood_data = gpd.read_file(r"C:\Users\vzhang\Documents\GitHub\Robotics\STARSWalkability\accesscore\data\Neighborhood_94_Count.geojson")
df = pd.read_csv(r"C:\Users\vzhang\Documents\GitHub\Robotics\STARSWalkability\accesscore\output\scored_edges_slope_access.csv")

In [11]:
# Neighborhood lookup function
def find_neighborhood(lat, lon):
    point = Point(lon, lat)
    match = neighborhood_data[neighborhood_data.contains(point)] 
    
    if not match.empty:
        return match.iloc[0]['S_HOOD']
    else: 
        return "No matching district found"

In [12]:
def parse_linestring(wkt_str):
    """
    Parses a WKT LINESTRING or MULTILINESTRING and returns a flat list of (lat, lon) tuples.

    Parameters:
        wkt_str (str): A WKT string in LINESTRING or MULTILINESTRING format.

    Returns:
        List[Tuple[float, float]]: List of (lat, lon) tuples.
    """
    wkt_str = wkt_str.strip()
    
    if wkt_str.startswith("LINESTRING"):
        coord_part = wkt_str[len("LINESTRING"):].strip(" ()")
        segments = [coord_part]
        
    elif wkt_str.startswith("MULTILINESTRING"):
        coord_part = wkt_str[len("MULTILINESTRING"):].strip(" ()")
        segments = coord_part.split("), (")
        
    else:
        raise ValueError("Input must start with 'LINESTRING' or 'MULTILINESTRING'")

    latlon_list = []

    for segment in segments:
        coords = segment.replace("(", "").replace(")", "").split(",")
        for pair in coords:
            lon_str, lat_str = pair.strip().split()
            latlon_list.append((float(lat_str), float(lon_str)))

    return latlon_list

In [13]:
neighborhoods = []

with alive_bar(len(df), force_tty=True) as bar:
    try:
        for index, row in df.iterrows():
            n_per_point = []
            pt_list = parse_linestring(row['geometry'])
            for tup in pt_list:
                n_per_point.append(find_neighborhood(tup[0], tup[1]))
            element_counts = Counter(n_per_point)
            neighborhoods.append(element_counts.most_common(1)[0][0])
            bar()
    except Exception as e:
        print("Partial neighborhoods:", neighborhoods)
        raise e

print("Finished Assigning Scores.")

|████████████████████████████████████████| 3189/3189 [100%] in 7.1s (446.62/s)   ▂▄▆ 494/3189 [15%] in 2s (~9s, 316.2/ ▇▇▅ 1542/3189 [48%] in 4s (~4s, 389.3 ▃▅▇ 2109/3189 [66%] in 5s (~3s, 417.2
Finished Assigning Scores.


In [ ]:
df['neighborhood'] = neighborhoods
df.to_csv(r"C:\Users\vzhang\Documents\GitHub\Robotics\STARSWalkability\accesscore\output\scored_edges.csv", index=False)

In [17]:
new_data = []
as_per_n = {}
df = pd.read_csv(r"C:\Users\vzhang\Documents\GitHub\Robotics\STARSWalkability\accesscore\output\scored_edges.csv")
with alive_bar(len(df), force_tty=True) as bar:
    try:
        for index, row in df.iterrows():
            if row['neighborhood'] in as_per_n.keys():
                as_per_n[row['neighborhood']].append(row['AccessScore_slope'])
            else:
                as_per_n[row['neighborhood']] = []
            bar()
        for val in as_per_n.keys():
            new_data.append({'neighborhood': val, 'avg_score': sum(as_per_n[val]) / len(as_per_n[val]), 'count': len(as_per_n[val])})
    except Exception as e:
       raise e
    
new_df = pd.DataFrame(new_data)
new_df.to_csv(r'C:\Users\vzhang\Documents\GitHub\Robotics\STARSWalkability\accesscore\output\neighborhood_scores.csv', index=False)

|████████████████████████████████████████| 3189/3189 [100%] in 0.1s (26193.86/s)
